# Version 2 : Random Survival Forest avec Features Moléculaires

**Objectif** : Intégrer les données de mutations génétiques pour améliorer la prédiction de survie.

**Améliorations par rapport à Version 1** :
- Ajout de ~30-50 features moléculaires basées sur les mutations génétiques
- Features de comptage de mutations (totales et par gène)
- Features d'agrégation de VAF (Variant Allele Frequency)
- Features par type d'effet (EFFECT)
- Features binaires pour les gènes les plus fréquents

**Features utilisées** :
- Toutes les features cliniques de Version 1
- Features moléculaires (comptages, VAF, EFFECT, présence de gènes)

**Architecture** : Même configuration RSF que Version 1

## 1. Import des bibliothèques

In [1]:
# Import des bibliothèques standards
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Survival analysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored, concordance_index_ipcw
from sksurv.util import Surv

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.inspection import permutation_importance

# Utilities
import warnings
warnings.filterwarnings('ignore')

# Configuration matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Bibliothèques importées avec succès")

✓ Bibliothèques importées avec succès


## 2. Chargement des données

In [2]:
# Chemins des fichiers
DATA_PATH = r"C:\Users\guill\Desktop\Data Challenge QRT\Data-Challenge-Prediction-de-Survie"

# Chargement des données cliniques
clinical_train = pd.read_csv(f"{DATA_PATH}\\X_train\\clinical_train.csv")
target_train = pd.read_csv(f"{DATA_PATH}\\target_train.csv")
clinical_test = pd.read_csv(f"{DATA_PATH}\\X_test\\clinical_test.csv")

# Chargement des données moléculaires
molecular_train = pd.read_csv(f"{DATA_PATH}\\X_train\\molecular_train.csv")
molecular_test = pd.read_csv(f"{DATA_PATH}\\X_test\\molecular_test.csv")

print(f"Clinical train shape: {clinical_train.shape}")
print(f"Molecular train shape: {molecular_train.shape}")
print(f"Target shape: {target_train.shape}")
print(f"Clinical test shape: {clinical_test.shape}")
print(f"Molecular test shape: {molecular_test.shape}")
print("\n✓ Données chargées")

Clinical train shape: (3323, 9)
Molecular train shape: (10935, 11)
Target shape: (3323, 3)
Clinical test shape: (1193, 9)
Molecular test shape: (3089, 11)

✓ Données chargées


## 3. Exploration des données moléculaires

In [3]:
print("="*60)
print("MOLECULAR DATA (Train)")
print("="*60)
display(molecular_train.head(10))

print("\n" + "="*60)
print("STATISTIQUES MOLÉCULAIRES")
print("="*60)
print(f"Nombre de mutations: {len(molecular_train)}")
print(f"Patients avec mutations: {molecular_train['ID'].nunique()}")
print(f"Gènes uniques: {molecular_train['GENE'].nunique()}")
print(f"Types d'effet (EFFECT): {molecular_train['EFFECT'].nunique()}")

print("\n" + "="*60)
print("TOP 20 GÈNES LES PLUS MUTÉS")
print("="*60)
top_20_genes = molecular_train['GENE'].value_counts().head(20)
print(top_20_genes)

print("\n" + "="*60)
print("DISTRIBUTION DES TYPES D'EFFET")
print("="*60)
print(molecular_train['EFFECT'].value_counts())

print("\n" + "="*60)
print("STATISTIQUES VAF")
print("="*60)
print(molecular_train['VAF'].describe())

MOLECULAR DATA (Train)


,ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH
0,P100000,11,119149248.0,119149248.0,G,A,CBL,p.C419Y,non_synonymous_codon,0.0830,1308.0
1,P100000,5,131822301.0,131822301.0,G,T,IRF1,p.Y164*,stop_gained,0.0220,532.0
2,P100000,3,77694060.0,77694060.0,G,C,ROBO2,p.?,splice_site_variant,0.4100,876.0
3,P100000,4,106164917.0,106164917.0,G,T,TET2,p.R1262L,non_synonymous_codon,0.4300,826.0
4,P100000,2,25468147.0,25468163.0,ACGAAGAGGGGGTGTTC,A,DNMT3A,p.E505fs*141,frameshift_variant,0.0898,942.0
5,P100001,22,29091725.0,29091725.0,C,T,CHEK2,p.W454*,stop_gained,0.4180,514.0
6,P100001,3,178936091.0,178936091.0,G,A,PIK3CA,p.E545K,non_synonymous_codon,0.1010,558.0
7,P100002,3,178936091.0,178936091.0,G,A,PIK3CA,p.E545K,non_synonymous_codon,0.1970,310.0
8,P100002,17,7579533.0,7579533.0,G,A,TP53,p.Q52*,stop_gained,0.5970,487.0
9,P100004,X,123190080.0,123190081.0,C,CA,STAG2,p.L436fs*5,frameshift_variant,0.4691,1296.0



STATISTIQUES MOLÉCULAIRES
Nombre de mutations: 10935
Patients avec mutations: 3026
Gènes uniques: 124
Types d'effet (EFFECT): 16

TOP 20 GÈNES LES PLUS MUTÉS
GENE
TET2      1663
ASXL1      951
SF3B1      775
DNMT3A     604
RUNX1      578
SRSF2      577
TP53       487
STAG2      376
U2AF1      288
EZH2       252
CBL        228
BCOR       213
NRAS       200
ZRSR2      196
DDX41      185
IDH2       166
CUX1       160
NF1        159
PHF6       149
KRAS       133
Name: count, dtype: int64

DISTRIBUTION DES TYPES D'EFFET
EFFECT
non_synonymous_codon            5471
frameshift_variant              2877
stop_gained                     1673
splice_site_variant              512
inframe_codon_loss               168
PTD                               89
inframe_codon_gain                55
ITD                               26
initiator_codon_change            24
2KB_upstream_variant              12
complex_change_in_transcript      11
3_prime_UTR_variant                5
inframe_variant            

## 4. Feature Engineering Moléculaire

### 4.1 Fonction de création des features moléculaires

In [4]:
def create_molecular_features(molecular_df, patient_ids, top_n_genes=20):
    """
    Crée les features moléculaires pour un ensemble de patients.
    
    Parameters:
    -----------
    molecular_df : DataFrame
        Données moléculaires avec colonnes ID, GENE, VAF, EFFECT
    patient_ids : list
        Liste des IDs de patients pour lesquels créer les features
    top_n_genes : int
        Nombre de gènes les plus fréquents à inclure
    
    Returns:
    --------
    DataFrame avec les features moléculaires pour chaque patient
    """
    
    # Créer un DataFrame vide avec tous les IDs patients
    mol_features = pd.DataFrame({'ID': patient_ids})
    
    # ===== 1. COMPTAGE DE MUTATIONS =====
    
    # Nombre total de mutations par patient
    mutation_counts = molecular_df.groupby('ID').size().to_frame('mutation_count_total')
    mol_features = mol_features.merge(mutation_counts, on='ID', how='left')
    
    # ===== 2. AGRÉGATION DE VAF =====
    
    # VAF moyenne, max et somme par patient
    vaf_stats = molecular_df.groupby('ID')['VAF'].agg([
        ('vaf_mean', 'mean'),
        ('vaf_max', 'max'),
        ('vaf_sum', 'sum')
    ]).reset_index()
    mol_features = mol_features.merge(vaf_stats, on='ID', how='left')
    
    # ===== 3. FEATURES PAR TYPE D'EFFET =====
    
    # Compter mutations par EFFECT
    effect_counts = molecular_df.groupby(['ID', 'EFFECT']).size().unstack(fill_value=0)
    effect_counts.columns = [f'effect_{col}' for col in effect_counts.columns]
    effect_counts = effect_counts.reset_index()
    mol_features = mol_features.merge(effect_counts, on='ID', how='left')
    
    # ===== 4. PRÉSENCE DE GÈNES SPÉCIFIQUES =====
    
    # Identifier les top N gènes les plus fréquents dans molecular_df
    top_genes_list = molecular_df['GENE'].value_counts().head(top_n_genes).index.tolist()
    
    # Compter mutations par gène (top N)
    for gene in top_genes_list:
        gene_mutations = molecular_df[molecular_df['GENE'] == gene].groupby('ID').size()
        mol_features[f'gene_{gene}_count'] = mol_features['ID'].map(gene_mutations)
    
    # Features binaires pour présence de gènes (top N)
    for gene in top_genes_list:
        gene_present = molecular_df[molecular_df['GENE'] == gene]['ID'].unique()
        mol_features[f'gene_{gene}_present'] = mol_features['ID'].isin(gene_present).astype(int)
    
    # ===== 5. IMPUTATION PAR 0 =====
    
    # Remplir toutes les valeurs NaN par 0 (absence de mutation)
    feature_cols = [col for col in mol_features.columns if col != 'ID']
    mol_features[feature_cols] = mol_features[feature_cols].fillna(0)
    
    # Retourner sans la colonne ID
    mol_features = mol_features.set_index('ID')
    
    print(f"✓ Features moléculaires créées: {mol_features.shape[1]} colonnes")
    print(f"  - Comptage mutations: 1 feature (total)")
    print(f"  - VAF agrégation: 3 features (mean, max, sum)")
    print(f"  - EFFECT types: {len([c for c in mol_features.columns if c.startswith('effect_')])} features")
    print(f"  - Gènes (comptages): {top_n_genes} features")
    print(f"  - Gènes (binaires): {top_n_genes} features")
    
    return mol_features

print("✓ Fonction create_molecular_features définie")

✓ Fonction create_molecular_features définie


### 4.2 Création des features moléculaires pour train

In [5]:
# Créer les features moléculaires pour l'ensemble d'entraînement
print("Création des features moléculaires (TRAIN)...")
print("="*60)

train_patient_ids = clinical_train['ID'].unique()
mol_features_train = create_molecular_features(molecular_train, train_patient_ids, top_n_genes=20)

print(f"\nShape finale: {mol_features_train.shape}")
print(f"Premières colonnes: {mol_features_train.columns[:5].tolist()}")

Création des features moléculaires (TRAIN)...
✓ Features moléculaires créées: 60 colonnes
  - Comptage mutations: 1 feature (total)
  - VAF agrégation: 3 features (mean, max, sum)
  - EFFECT types: 16 features
  - Gènes (comptages): 20 features
  - Gènes (binaires): 20 features

Shape finale: (3323, 60)
Premières colonnes: ['mutation_count_total', 'vaf_mean', 'vaf_max', 'vaf_sum', 'effect_2KB_upstream_variant']


## 5. Intégration avec les features cliniques

In [6]:
# Nettoyer la target
target_clean = target_train.dropna(subset=['OS_YEARS', 'OS_STATUS']).copy()
target_clean['OS_STATUS'] = target_clean['OS_STATUS'].astype(bool)
target_clean = target_clean.set_index('ID')

# Filtrer clinical_train pour garder seulement les IDs présents dans target_clean
clinical_train_clean = clinical_train[clinical_train['ID'].isin(target_clean.index)].copy()
clinical_train_clean = clinical_train_clean.set_index('ID')
clinical_train_clean = clinical_train_clean.loc[target_clean.index]

print(f"✓ Données nettoyées: {len(clinical_train_clean)} patients")
print(f"  Décès: {target_clean['OS_STATUS'].sum()}")
print(f"  Censurés: {(~target_clean['OS_STATUS']).sum()}")

✓ Données nettoyées: 3173 patients
  Décès: 1600
  Censurés: 1573


In [7]:
# Sélection des features cliniques numériques
numeric_features = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'HB', 'PLT']
X_numeric = clinical_train_clean[numeric_features].copy()

# Encodage de CENTER
center_encoded = pd.get_dummies(clinical_train_clean['CENTER'], prefix='CENTER', drop_first=True)

# Concaténation des features cliniques
X_clinical = pd.concat([X_numeric, center_encoded], axis=1)

print(f"✓ Features cliniques: {X_clinical.shape[1]} colonnes")
print(f"  Numériques: {len(numeric_features)}")
print(f"  CENTER encodé: {center_encoded.shape[1]}")

✓ Features cliniques: 28 colonnes
  Numériques: 6
  CENTER encodé: 22


In [8]:
# Aligner les features moléculaires avec X_clinical
mol_features_train_aligned = mol_features_train.reindex(X_clinical.index, fill_value=0)

# Combiner features cliniques et moléculaires
X_combined = pd.concat([X_clinical, mol_features_train_aligned], axis=1)

print("="*60)
print("FEATURES COMBINÉES (CLINICAL + MOLECULAR)")
print("="*60)
print(f"Total features: {X_combined.shape[1]}")
print(f"  Clinical: {X_clinical.shape[1]}")
print(f"  Molecular: {mol_features_train_aligned.shape[1]}")
print(f"\nPatients: {X_combined.shape[0]}")
print(f"\nPremières features moléculaires:")
print(mol_features_train_aligned.columns[:10].tolist())

FEATURES COMBINÉES (CLINICAL + MOLECULAR)
Total features: 88
  Clinical: 28
  Molecular: 60

Patients: 3173

Premières features moléculaires:
['mutation_count_total', 'vaf_mean', 'vaf_max', 'vaf_sum', 'effect_2KB_upstream_variant', 'effect_3_prime_UTR_variant', 'effect_ITD', 'effect_PTD', 'effect_complex_change_in_transcript', 'effect_frameshift_variant']


## 6. Imputation des valeurs manquantes

In [9]:
# Imputation des valeurs manquantes par la médiane
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(
    imputer.fit_transform(X_combined),
    index=X_combined.index,
    columns=X_combined.columns
)

print("✓ Imputation terminée")
print(f"  Valeurs manquantes restantes: {X_imputed.isnull().sum().sum()}")
print(f"  Shape: {X_imputed.shape}")

✓ Imputation terminée
  Valeurs manquantes restantes: 0
  Shape: (3173, 88)


In [10]:
# Création du format survival pour sksurv
y_surv = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_clean)

print(f"✓ Target survival créée: {len(y_surv)} observations")
print(f"  Type: {type(y_surv)}")
print(f"  Événements: {y_surv['OS_STATUS'].sum()}")

✓ Target survival créée: 3173 observations
  Type: <class 'numpy.ndarray'>
  Événements: 1600


## 7. Split Train/Validation

In [11]:
# Split 70/30 avec stratification sur OS_STATUS
X_train, X_val, y_train, y_val = train_test_split(
    X_imputed, 
    y_surv,
    test_size=0.3,
    random_state=42,
    stratify=target_clean['OS_STATUS'].astype(int)
)

print("="*60)
print("SPLIT TRAIN/VALIDATION")
print("="*60)
print(f"Train set: {X_train.shape[0]} patients ({X_train.shape[0]/len(X_imputed)*100:.1f}%)")
print(f"  - Features: {X_train.shape[1]}")
print(f"  - Décès: {y_train['OS_STATUS'].sum()}")
print(f"  - Censurés: {(~y_train['OS_STATUS']).sum()}")
print(f"  - Taux de censure: {(~y_train['OS_STATUS']).sum() / len(y_train) * 100:.1f}%")

print(f"\nValidation set: {X_val.shape[0]} patients ({X_val.shape[0]/len(X_imputed)*100:.1f}%)")
print(f"  - Features: {X_val.shape[1]}")
print(f"  - Décès: {y_val['OS_STATUS'].sum()}")
print(f"  - Censurés: {(~y_val['OS_STATUS']).sum()}")
print(f"  - Taux de censure: {(~y_val['OS_STATUS']).sum() / len(y_val) * 100:.1f}%")

SPLIT TRAIN/VALIDATION
Train set: 2221 patients (70.0%)
  - Features: 88
  - Décès: 1120
  - Censurés: 1101
  - Taux de censure: 49.6%

Validation set: 952 patients (30.0%)
  - Features: 88
  - Décès: 480
  - Censurés: 472
  - Taux de censure: 49.6%


## 8. Entraînement du Random Survival Forest

In [12]:
# Configuration du modèle RSF (même architecture que Version 1)
rsf = RandomSurvivalForest(
    n_estimators=100,        # Nombre d'arbres
    min_samples_split=10,    # Régularisation
    min_samples_leaf=15,     # Régularisation
    max_features="sqrt",     # Nombre de features à considérer par split
    n_jobs=-1,               # Utiliser tous les CPU
    random_state=42,
    verbose=1
)

print("="*60)
print("CONFIGURATION DU MODÈLE")
print("="*60)
print(f"Nombre d'arbres: {rsf.n_estimators}")
print(f"Min samples split: {rsf.min_samples_split}")
print(f"Min samples leaf: {rsf.min_samples_leaf}")
print(f"Max features: {rsf.max_features}")
print(f"\n🌲 Entraînement en cours...")

CONFIGURATION DU MODÈLE
Nombre d'arbres: 100
Min samples split: 10
Min samples leaf: 15
Max features: sqrt

🌲 Entraînement en cours...


In [13]:
# Entraînement du modèle
rsf.fit(X_train, y_train)

print("\n✓ Modèle entraîné avec succès!")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.3s



✓ Modèle entraîné avec succès!


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.9s finished


## 9. Évaluation du modèle

In [14]:
# Prédiction des scores de risque
y_pred_train = rsf.predict(X_train)
y_pred_val = rsf.predict(X_val)

# Calcul du C-index
c_index_train = concordance_index_censored(
    y_train['OS_STATUS'], 
    y_train['OS_YEARS'], 
    y_pred_train
)[0]

c_index_val = concordance_index_censored(
    y_val['OS_STATUS'], 
    y_val['OS_YEARS'], 
    y_pred_val
)[0]

print("="*60)
print("PERFORMANCE DU MODÈLE (C-INDEX)")
print("="*60)
print(f"Train C-index: {c_index_train:.4f}")
print(f"Validation C-index: {c_index_val:.4f}")
print(f"\nDifférence (overfitting): {c_index_train - c_index_val:.4f}")

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.7s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    2.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.1s


PERFORMANCE DU MODÈLE (C-INDEX)
Train C-index: 0.7815
Validation C-index: 0.7334

Différence (overfitting): 0.0481


[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.6s finished


## 10. Importance des features

In [ ]:
# Calcul de l'importance des features (basé sur la permutation)
print("Calcul de l'importance des features (peut prendre quelques minutes)...")

perm_importance = permutation_importance(
    rsf, X_val, y_val,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

# Créer un DataFrame avec les importances
feature_importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': perm_importance.importances_mean,
    'std': perm_importance.importances_std
}).sort_values('importance', ascending=False)

print("\n✓ Importance calculée")
print("\n" + "="*60)
print("TOP 20 FEATURES LES PLUS IMPORTANTES")
print("="*60)
display(feature_importance_df.head(20))

Calcul de l'importance des features (peut prendre quelques minutes)...


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.2s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.8s finished


In [ ]:
# Visualisation de l'importance des features
plt.figure(figsize=(12, 8))
top_n = 20
top_features = feature_importance_df.head(top_n)

plt.barh(range(top_n), top_features['importance'], xerr=top_features['std'])
plt.yticks(range(top_n), top_features['feature'])
plt.xlabel('Importance (permutation)')
plt.title(f'Top {top_n} Features les plus importantes')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\n✓ Visualisation créée")

## 11. Analyse comparative des features moléculaires

In [ ]:
# Séparer les features cliniques et moléculaires
clinical_feature_names = X_clinical.columns.tolist()
molecular_feature_names = mol_features_train_aligned.columns.tolist()

# Importance moyenne pour chaque type de feature
clinical_importance = feature_importance_df[
    feature_importance_df['feature'].isin(clinical_feature_names)
]['importance']

molecular_importance = feature_importance_df[
    feature_importance_df['feature'].isin(molecular_feature_names)
]['importance']

print("="*60)
print("COMPARAISON IMPORTANCE: CLINICAL vs MOLECULAR")
print("="*60)
print(f"\nImportance moyenne Clinical features: {clinical_importance.mean():.6f}")
print(f"Importance moyenne Molecular features: {molecular_importance.mean():.6f}")

print(f"\nTop 10 features - Breakdown:")
top_10 = feature_importance_df.head(10)
print(f"  Clinical: {sum(top_10['feature'].isin(clinical_feature_names))}")
print(f"  Molecular: {sum(top_10['feature'].isin(molecular_feature_names))}")

## 12. Préparation du test set et prédictions

In [ ]:
# Créer les features moléculaires pour le test set
print("Création des features moléculaires (TEST)...")
print("="*60)

test_patient_ids = clinical_test['ID'].unique()
mol_features_test = create_molecular_features(molecular_test, test_patient_ids, top_n_genes=20)

print(f"\nShape: {mol_features_test.shape}")

In [ ]:
# Préparer les features cliniques pour le test set
clinical_test_indexed = clinical_test.set_index('ID')

# Features numériques
X_numeric_test = clinical_test_indexed[numeric_features].copy()

# Encodage de CENTER
center_encoded_test = pd.get_dummies(clinical_test_indexed['CENTER'], prefix='CENTER', drop_first=True)

# S'assurer que les colonnes CENTER sont les mêmes que dans le train
for col in center_encoded.columns:
    if col not in center_encoded_test.columns:
        center_encoded_test[col] = 0

center_encoded_test = center_encoded_test[center_encoded.columns]

# Combiner les features cliniques
X_clinical_test = pd.concat([X_numeric_test, center_encoded_test], axis=1)

# Aligner les features moléculaires
mol_features_test_aligned = mol_features_test.reindex(X_clinical_test.index, fill_value=0)

# Combiner clinical + molecular
X_test_combined = pd.concat([X_clinical_test, mol_features_test_aligned], axis=1)

# S'assurer que les colonnes sont les mêmes que dans le train
for col in X_combined.columns:
    if col not in X_test_combined.columns:
        X_test_combined[col] = 0

X_test_combined = X_test_combined[X_combined.columns]

print(f"✓ Features test préparées")
print(f"  Shape: {X_test_combined.shape}")
print(f"  Colonnes alignées: {X_test_combined.shape[1] == X_combined.shape[1]}")

In [ ]:
# Imputation pour le test set avec le même imputer
X_test_imputed = pd.DataFrame(
    imputer.transform(X_test_combined),
    index=X_test_combined.index,
    columns=X_test_combined.columns
)

print(f"✓ Test set imputé")
print(f"  Valeurs manquantes: {X_test_imputed.isnull().sum().sum()}")

## 13. Prédictions finales et génération du fichier de soumission

In [ ]:
# Ré-entraîner le modèle sur toutes les données d'entraînement
print("Ré-entraînement du modèle sur l'ensemble complet des données...")
rsf_final = RandomSurvivalForest(
    n_estimators=100,
    min_samples_split=10,
    min_samples_leaf=15,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42,
    verbose=1
)

rsf_final.fit(X_imputed, y_surv)
print("\n✓ Modèle final entraîné")

In [ ]:
# Prédiction sur le test set
predictions_test = rsf_final.predict(X_test_imputed)

# Créer le DataFrame de soumission
submission = pd.DataFrame({
    'ID': X_test_imputed.index,
    'PRED': predictions_test
})

# Sauvegarder le fichier de soumission
submission_path = f"{DATA_PATH}\\submission_rsf_molecular.csv"
submission.to_csv(submission_path, index=False)

print("="*60)
print("SOUMISSION GÉNÉRÉE")
print("="*60)
print(f"Fichier: {submission_path}")
print(f"Nombre de prédictions: {len(submission)}")
print(f"\nPremières prédictions:")
display(submission.head(10))

print(f"\nStatistiques des prédictions:")
print(submission['PRED'].describe())

## 14. Résumé des résultats

In [ ]:
print("="*60)
print("RÉSUMÉ - VERSION 2: RSF avec Features Moléculaires")
print("="*60)

print("\n📊 FEATURES UTILISÉES:")
print(f"  • Total features: {X_combined.shape[1]}")
print(f"    - Clinical: {X_clinical.shape[1]} features")
print(f"    - Molecular: {mol_features_train_aligned.shape[1]} features")

print("\n🧬 BREAKDOWN FEATURES MOLÉCULAIRES:")
print(f"  • Comptage mutations totales: 1")
print(f"  • Agrégation VAF (mean, max, sum): 3")
print(f"  • Types d'effet (EFFECT): ~{len([c for c in mol_features_train_aligned.columns if c.startswith('effect_')])}")
print(f"  • Comptages par gène (top 20): 20")
print(f"  • Présence gène binaire (top 20): 20")

print("\n📈 PERFORMANCE:")
print(f"  • Train C-index: {c_index_train:.4f}")
print(f"  • Validation C-index: {c_index_val:.4f}")

print("\n🏆 TOP 5 FEATURES:")
for i, row in feature_importance_df.head(5).iterrows():
    feature_type = "Clinical" if row['feature'] in clinical_feature_names else "Molecular"
    print(f"  {i+1}. {row['feature']:<30} ({feature_type}): {row['importance']:.6f}")

print("\n✓ Analyse terminée!")